# Training Notebook
This notebook runs local queue-environment smoke tests, PPO training, and the combined benchmark using the project modules.


In [5]:
!git clone https://github.com/DylanB03/Job-Scheduler.git
%cd Job-Scheduler

Cloning into 'Job-Scheduler'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 111 (delta 34), reused 103 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 616.87 KiB | 18.69 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/Job-Scheduler


In [9]:
!pip install .

Processing /content/Job-Scheduler
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [6]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "baselines").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: /content/Job-Scheduler


In [7]:
from config.base_config import hyperparams_config

cfg = hyperparams_config()
cfg

{'total_timesteps': 500000,
 'episodes': 50,
 'max_jobs': 10,
 'max_steps': None,
 'learning_rate': 0.0003,
 'gamma': 0.99,
 'gae_lambda': 0.95,
 'clip_coef': 0.2,
 'ent_coef': 0.01,
 'vf_coef': 0.5,
 'update_epochs': 10,
 'num_minibatches': 4,
 'max_grad_norm': 0.5,
 'kl_target': 0.015,
 'device': 'cuda',
 'cuda': True,
 'seed': 0,
 'output_dir': PosixPath('/content/Job-Scheduler/results')}

## Optional: Heuristic Benchmark Smoke Test
Run this first to verify env behavior quickly before PPO.

In [8]:
from envs.queue_env import QueueEnv
from baselines.heuristics import build_default_baselines, evaluate_baselines

env = QueueEnv(max_jobs=cfg["max_jobs"], max_steps=cfg["max_steps"])
results = evaluate_baselines(
    build_default_baselines(),
    env,
    n_episodes=cfg["episodes"],
    base_seed=cfg["seed"],
)

for r in results:
    print(r["name"], "reward=", round(r["mean_reward"], 2), "drops=", round(r["mean_dropped_jobs"], 2))

ModuleNotFoundError: No module named 'stable_baselines3'

## Train PPO
By default this uses `total_timesteps` from `config/config.ini`.

In [ ]:
from baselines.train_ppo import PPOTrainer

trainer = PPOTrainer()

# Optional quick test run: uncomment to shorten training
# trainer.config["total_timesteps"] = 50_000

trainer.train()
saved_path = trainer.save("ppo_queue_notebook")
print("Saved PPO model to", saved_path)

## Optional: Train Custom PPO
Run this only if you want to compare your own PPO implementation against SB3 and the heuristics.


In [ ]:
from baselines.ppo_agent import PPOAgent

custom_trainer = PPOAgent()

# Optional quick test run: uncomment to shorten training
# custom_trainer.config["total_timesteps"] = 50_000

custom_trainer.train()
custom_saved_path = custom_trainer.save("custom_ppo_notebook")
print("Saved custom PPO model to", custom_saved_path)


## Evaluate and Benchmark
Load any saved PPO models and compare them against the heuristic baselines.


In [ ]:
from baselines.eval_all import main

custom_path = custom_saved_path if "custom_saved_path" in globals() else None

main(
    train_ppo=False,
    model_path=saved_path,
    train_custom_ppo=False,
    custom_model_path=custom_path,
)
